# QRC resonator spectroscopy (readout only)

This example targets the QRC in slot 12 at cluster address `192.168.0.2`, using the same cluster-level address style as the existing QRM-RF platform. The Qibolab platform connects to `192.168.0.2` without a slot suffix; slot 12 is selected by the channel paths. The platform is configured for QRC paired IO port 1 (`io1` in the platform map, corresponding to Qblox `out0_in0`). If the resonator cable is on paired IO port 2, change the platform mapping and LO name to `io2` and `qrc/out1_in1`.

**Do not run the hardware cells until you have confirmed the pair and cabling, entered calibrated values in `parameters.json`, and checked the QRC input attenuation and safe output level with your lab setup.** Hardware execution is disabled by default. The `0.0` frequency values in the runcard are placeholders, not hardware-ready settings.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qibolab import (
    Acquisition,
    AcquisitionType,
    Gaussian,
    Parameter,
    Pulse,
    PulseSequence,
    Readout,
    Sweeper,
    create_platform,
)

platform_name = "qblox_qrc_readout"
if "QIBOLAB_PLATFORMS" not in os.environ:
    roots = [Path.cwd(), *Path.cwd().parents]
    platforms_dir = next(
        (
            root / "platforms"
            for root in roots
            if (root / "platforms" / platform_name / "platform.py").is_file()
        ),
        None,
    )
    if platforms_dir is None:
        raise RuntimeError(
            "Set QIBOLAB_PLATFORMS to the absolute path of the platforms directory."
        )
    os.environ["QIBOLAB_PLATFORMS"] = str(platforms_dir.resolve())

platform = create_platform(platform_name)
qubit = platform.qubits["q0"]
lo_name = "qrc/out0_in0"
print("Platform:", platform.name)
print("Controller address:", platform.instruments["cluster0"].address)
print("Probe:", qubit.probe, platform.channels[qubit.probe])
print("Acquisition:", qubit.acquisition, platform.channels[qubit.acquisition])
for name in (qubit.probe, qubit.acquisition, lo_name):
    print(f"{name}: {platform.parameters.configs[name]}")

## 1. Prepare and review calibrated settings

Edit `platforms/qblox_qrc_readout/parameters.json` before hardware use:

- Set `q0/probe.frequency` to a reasonable resonator-frequency estimate from your device design or a prior measurement (Hz).
- Set `qrc/out0_in0.frequency` to the QRC paired-port center/LO frequency (Hz). For the QRC paired center-frequency setting, use the hardware-supported 500 MHz–10.1 GHz range and 100 MHz increments. The probe frequency is then set relative to this center using the sequencer NCO.
- Set the acquisition delay and smearing from the readout timing calibration. Check the RF path attenuation at the QRC input and the readout output level with the lab owner; the oscillator `power` field is not a substitute for checking physical attenuation and safe output level.

Use the actual QRC input/output pair to which the resonator is cabled. Keep the span small for the first scan; only widen it after confirming the readout path is behaving safely.

In [ ]:
# Replace these with values cleared for this setup. Units: ns and normalized amplitude.
READOUT_DURATION_NS = 1000
READOUT_AMPLITUDE = 0.001
N_SHOTS = 256
SCAN_SPAN_HZ = 10e6
N_FREQUENCIES = 51
CALIBRATION_REVIEWED = True
RUN_ONE_POINT = True
RUN_SPECTROSCOPY = True

probe_config = platform.parameters.configs[qubit.probe]
acquisition_config = platform.parameters.configs[qubit.acquisition]
lo_config = platform.parameters.configs[lo_name]

assert CALIBRATION_REVIEWED, "Review the wiring and calibrated values first."
assert probe_config.frequency > 0, "Set a calibrated resonator-frequency estimate."
assert 500e6 <= lo_config.frequency <= 10.1e9, "Set a valid QRC center frequency."
assert np.isclose(lo_config.frequency % 100e6, 0), (
    "The QRC paired center frequency must be on a 100 MHz increment."
)
assert READOUT_DURATION_NS > 0
assert 0 < abs(READOUT_AMPLITUDE) <= 1
assert N_SHOTS > 0 and N_FREQUENCIES >= 3 and SCAN_SPAN_HZ > 0
print("Calibration checks passed.")
print(
    "Acquisition delay/smearing:", acquisition_config.delay, acquisition_config.smearing
)

In [ ]:
probe_pulse = Pulse(
    duration=READOUT_DURATION_NS,
    amplitude=READOUT_AMPLITUDE,
    relative_phase=0.0,
    envelope=Gaussian(rel_sigma=0.2),
)
sequence = PulseSequence(
    [
        (
            qubit.acquisition,
            Readout(
                probe=probe_pulse,
                acquisition=Acquisition(duration=READOUT_DURATION_NS),
            ),
        )
    ]
)
readout_id = sequence.acquisitions[0][1].id
print(f"Prepared one {READOUT_DURATION_NS} ns readout pulse/acquisition.")

In [ ]:
from qblox_instruments import Cluster
from qcodes.instrument import find_or_create_instrument

cluster = find_or_create_instrument(
    Cluster,
    recreate=True,
    name="qrc_route_diagnostic",
    identifier="192.168.0.2",  # keep the cluster-level address
)

try:
    qrc_modules = cluster.get_connected_modules(lambda mod: mod.is_qrc_type)
    module = next(mod for mod in qrc_modules.values() if mod.slot_idx == 12)
    print("Found QRC in slot", module.slot_idx)

    # The QRC tutorial uses 20 dB input and output attenuation.
    module.out0_att(20)
    module.in0_att(20)
    module.out0_in0_lo_freq(int(lo_config.frequency))
    print("Attenuation and center frequency configured")

    module.disconnect_outputs()
    print("disconnect_outputs succeeded")
    module.disconnect_inputs()
    print("disconnect_inputs succeeded")

    module.sequencer0.connect_out0("IQ")
    print("connect_out0 succeeded")
    module.sequencer0.connect_acq("in0")
    print("connect_acq succeeded")

    print(module.get_sequencer_status(0))

    module.stop_sequencer()  # baseline

    module.disconnect_outputs()
    module.stop_sequencer()

    module.disconnect_inputs()
    module.stop_sequencer()

    module.sequencer0.connect_out0("IQ")
    module.stop_sequencer()

    module.sequencer0.connect_acq("in0")
    module.stop_sequencer()
finally:
    cluster.close()

In [ ]:
import sys


def show_error(error, indent=""):
    print(f"{indent}{type(error).__name__}: {error}")
    if isinstance(error, BaseExceptionGroup):
        for child in error.exceptions:
            show_error(child, indent + "  ")
    elif error.__cause__ is not None:
        show_error(error.__cause__, indent + "  ")


show_error(sys.last_value)

## 2. Verify a single readout point first

After reviewing the calibration and wiring, enable only `RUN_ONE_POINT`. The cell executes one acquisition, calls `stop_sequencer()` on each detected module as a diagnostic, then runs normal platform teardown (which also stops the modules). It reports the pre-teardown stop result separately and prints the full nested teardown error. Do not enable the sweep until one point and cleanup both succeed.

In [ ]:
def print_exception_tree(error, indent=""):
    print(f"{indent}{type(error).__name__}: {error}")
    if isinstance(error, BaseExceptionGroup):
        for child in error.exceptions:
            print_exception_tree(child, indent + "  ")
    elif error.__cause__ is not None:
        print_exception_tree(error.__cause__, indent + "  ")


def execute_and_disconnect(*args, check_stop_before_disconnect=False, **kwargs):
    platform.connect()
    controller = platform.instruments["cluster0"]
    execution_error = None
    stop_error = None
    cleanup_error = None
    results = None
    try:
        try:
            results = platform.execute(*args, **kwargs)
        except Exception as error:
            execution_error = error
        if execution_error is None and check_stop_before_disconnect:
            for slot in sorted(controller._channels_by_module):
                module = controller._modules[slot]
                try:
                    module.stop_sequencer()
                    print(f"Pre-disconnect stop succeeded for slot {slot}.")
                except Exception as error:
                    stop_error = error
                    print(f"Pre-disconnect stop failed for slot {slot}: {error}")
                    break
    finally:
        try:
            platform.disconnect()
        except Exception as error:
            cleanup_error = error
            print("Disconnect raised:")
            print_exception_tree(error)

    failures = [
        ("Execution", execution_error),
        ("Pre-disconnect stop check", stop_error),
        ("Disconnect", cleanup_error),
    ]
    failures = [(stage, error) for stage, error in failures if error is not None]
    if len(failures) > 1:
        raise ExceptionGroup(
            "; ".join(f"{stage} failed" for stage, _ in failures),
            [error for _, error in failures],
        )
    if failures:
        stage, error = failures[0]
        raise RuntimeError(f"{stage} failed; results must not be used.") from error
    return results


if not RUN_ONE_POINT:
    print("Skipped. Set RUN_ONE_POINT = True only after reviewing the setup.")
else:
    assert CALIBRATION_REVIEWED
    point_results = execute_and_disconnect(
        [sequence],
        nshots=N_SHOTS,
        acquisition_type=AcquisitionType.INTEGRATION,
        check_stop_before_disconnect=True,
    )
    point_iq = np.asarray(point_results[readout_id], dtype=float)
    if point_iq.shape != (N_SHOTS, 2) or not np.isfinite(point_iq).all():
        raise ValueError(
            f"Expected finite ({N_SHOTS}, 2) I/Q values; got {point_iq.shape}."
        )
    print("Mean I/Q:", point_iq.mean(axis=0))
    print("I/Q standard deviation:", point_iq.std(axis=0))

## 3. Sweep the probe frequency

Once a single point succeeds, set `RUN_ONE_POINT = False`, set `RUN_SPECTROSCOPY = True`, and rerun the parameter, sequence, and sweep cells. The scan is centered on the `q0/probe.frequency` estimate and uses a 10 MHz total span by default. Choose a center near the expected resonance and keep the first span modest. Each frequency is acquired for `N_SHOTS`; results are averaged over shots before plotting.

In [ ]:
center_hz = probe_config.frequency
frequencies = np.linspace(
    center_hz - SCAN_SPAN_HZ / 2,
    center_hz + SCAN_SPAN_HZ / 2,
    N_FREQUENCIES,
)
assert frequencies.min() > 0, "The frequency scan must remain above 0 Hz."
frequency_sweeper = Sweeper(
    parameter=Parameter.frequency,
    values=frequencies,
    channels=[qubit.probe],
)

if not RUN_SPECTROSCOPY:
    print("Skipped. Set RUN_SPECTROSCOPY = True to run the frequency sweep.")
else:
    assert CALIBRATION_REVIEWED
    spectroscopy_results = execute_and_disconnect(
        [sequence],
        [[frequency_sweeper]],
        nshots=N_SHOTS,
        acquisition_type=AcquisitionType.INTEGRATION,
    )

    iq = np.asarray(spectroscopy_results[readout_id], dtype=float)
    expected_shape = (N_SHOTS, N_FREQUENCIES, 2)
    if iq.shape != expected_shape or not np.isfinite(iq).all():
        raise ValueError(
            f"Expected finite results with shape {expected_shape}; got {iq.shape}."
        )
    mean_iq = iq.mean(axis=0)
    response = mean_iq[:, 0] + 1j * mean_iq[:, 1]
    print(f"Acquired {N_FREQUENCIES} frequencies, {N_SHOTS} shots each.")

In [ ]:
if RUN_SPECTROSCOPY and "response" in globals():
    fig, (ax_mag, ax_iq, ax_complex) = plt.subplots(3, 1, figsize=(8, 10))
    ax_mag.plot(frequencies / 1e9, np.abs(response), ".-")
    ax_mag.set_ylabel("|I + iQ|")
    ax_mag.set_title("QRC resonator spectroscopy")
    ax_mag.grid(True, alpha=0.3)

    ax_iq.plot(frequencies / 1e9, mean_iq[:, 0], label="I")
    ax_iq.plot(frequencies / 1e9, mean_iq[:, 1], label="Q")
    ax_iq.set_ylabel("Mean integrated response")
    ax_iq.legend()
    ax_iq.grid(True, alpha=0.3)

    points = ax_complex.scatter(
        response.real, response.imag, c=frequencies / 1e9, cmap="viridis"
    )
    fig.colorbar(points, ax=ax_complex, label="Probe frequency (GHz)")
    ax_complex.set_xlabel("I")
    ax_complex.set_ylabel("Q")
    ax_complex.set_title("Complex I/Q trajectory across the sweep")
    ax_complex.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
    print("Largest response at (GHz):", frequencies[np.argmax(np.abs(response))] / 1e9)
else:
    print("No spectroscopy data to plot yet.")

## How to tell whether it worked

- **Connection and one-point acquisition:** no connection/compatibility error, and finite I/Q samples are returned. This verifies communication and acquisition, but not resonator alignment.
- **Spectrum:** look for a repeatable change in magnitude and/or an I/Q rotation/arc near the resonator frequency. A smooth trace with no feature may mean the scan missed the resonance, the wrong IO pair is selected, or the RF path is not connected; check wiring and calibration before widening the span.
- **Flat, noisy, or saturated trace:** verify the chosen QRC pair, LO and probe frequencies, acquisition delay, cable path, and input/output attenuation. If the response is saturated, stop and lower the output level or increase attenuation according to the lab procedure.
- Repeat a narrow scan around the feature with smaller steps. Save the frequency axis and `mean_iq` so you can compare the feature between runs. Do not treat this readout-only scan as qubit spectroscopy; it probes the resonator with a fixed pulse amplitude.